In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Get_engine_forcefield)=
# Get engine forcefield

*Translating force field specifications into engine-specific parameter definitions.*

The function {func}`molsysmt.molecular_mechanics.get_engine_forcefield` returns an independent, ordered list of resource names: OpenMM XML files or LEaP leaprc files. Looking up these unit-free strings does not import the engine, find its files, construct a force field, or parameterize a system.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.molecular_mechanics.get_engine_forcefield`.
:::


Stored constraint policies describe physical intent. The native legacy
`get_openmm_System_parameters()` helper and public conversion share the
`none`/`hbonds`/`all bonds`/`h angles` translation, including underscore spellings;
`all_bonds` does not mean hydrogen-only bonds. A returned legacy keyword
dictionary does not execute a model or certify support by every constructor.
See [the native mechanics domain](../../foundations/native_world/classes/molsysmt_MolecularMechanics.md)
for the remaining requested-versus-effective recipe limits.


:::{note}
:class: dropdown

**Molecular Mechanics Backend Engine**

Molecular mechanics and force evaluations in MolSysMT are executed using backend simulation engines (currently **OpenMM** by default). MolSysMT intentionally unifies the Pythonic interface and data structures while delegating energy evaluations, parameterizations, and gradient calculations to high-performance, battle-tested molecular simulation engines.
:::


## Basic usage

Let's query the OpenMM XML definition files for the AMBER14 force field in vacuum:


In [2]:
import molsysmt as msm


In [3]:
ff_xmls = msm.molecular_mechanics.get_engine_forcefield('AMBER14', engine='OpenMM')
print('OpenMM XML definitions for AMBER14 in vacuum:', ff_xmls)


OpenMM XML definitions for AMBER14 in vacuum: ['amber14-all.xml']


## Selecting solvent labels

An explicit water model selects its registered resource list. An implicit-solvent label takes precedence when both arguments are supplied. OBC1 and OBC2 identify different physical models. The OpenMM mapping selects their separate standard model XMLs; a LEaP lookup only prepares resource names and does not configure a subsequent simulation's igb option. Resource lookup alone does not execute either model. Omitting both arguments selects vacuum.


In [4]:
ff_tip3p = msm.molecular_mechanics.get_engine_forcefield('AMBER14', water_model='TIP3P', engine='OpenMM')
ff_obc1 = msm.molecular_mechanics.get_engine_forcefield('AMBER14', implicit_solvent='OBC1', engine='OpenMM')

print('AMBER14 + TIP3P XMLs:', ff_tip3p)
print('AMBER14 + OBC1 XMLs:', ff_obc1)


AMBER14 + TIP3P XMLs: ['amber14-all.xml', 'amber14/tip3p.xml']
AMBER14 + OBC1 XMLs: ['amber14-all.xml', 'implicit/obc1.xml']


## Looking up another engine

LEaP mappings return leaprc names. The engine need not be installed to inspect this metadata; actual execution checks the required engine and resources.

In [5]:
leap_resources = msm.molecular_mechanics.get_engine_forcefield('AMBER14', water_model='TIP3P', engine='LEaP')
print('LEaP resource names:', leap_resources)

LEaP resource names: ['leaprc.protein.ff14SB', 'leaprc.water.tip3p']


## Choosing a solvent model

When both valid solvent arguments are supplied, the implicit-solvent entry wins. Not every engine, force field and model combination has an entry; an unregistered combination raises a native argument diagnostic.

In [6]:
preferred_resources = msm.molecular_mechanics.get_engine_forcefield('AMBER10', implicit_solvent='OBC1', water_model='TIP3P')
print('Implicit-solvent entry:', preferred_resources)

Implicit-solvent entry: ['amber10.xml', 'implicit/obc1.xml']


## Editing a local resource list

Each call returns its own list. You can edit it for a separate engine workflow without changing the registry, other results, or later MolSysMT parameterizations. Such edits do not configure a subsequent MolSysMT calculation; using a custom list requires passing it to the engine explicitly.

In [7]:
local_resources = msm.molecular_mechanics.get_engine_forcefield('AMBER14', water_model='TIP3P')
local_resources.append('my_extra_parameters.xml')
fresh_resources = msm.molecular_mechanics.get_engine_forcefield('AMBER14', water_model='TIP3P')
assert fresh_resources == ['amber14-all.xml', 'amber14/tip3p.xml']
print('Local resources:', local_resources)
print('Independent lookup:', fresh_resources)

Local resources: ['amber14-all.xml', 'amber14/tip3p.xml', 'my_extra_parameters.xml']
Independent lookup: ['amber14-all.xml', 'amber14/tip3p.xml']


## Keeping the physical model explicit

The solute field and solvent model are independent, engine-neutral settings. Declare them on the molecular system; supported stored settings precede conversion defaults, while explicit keywords override them. The example chooses OpenMM as the execution engine through the output form. OBC1 and OBC2 use OpenMM's standard mbondi2 radii and ACE term, dielectric constants 1/78.5 and zero screening. Independent dielectric/screening/cutoff tuning is not yet exposed by this converter.

This delegates the published Onufriev-Bashford-Case method to OpenMM. It does not implement a new GB method. See [the scientific reference](https://doi.org/10.1002/prot.20033) and [the engine definitions](https://docs.openmm.org/latest/userguide/application/02_running_sims.html#implicit-solvent).

In [8]:
molsys = msm.convert(msm.systems['Trp-Cage']['1l2y.h5msm'], structure_indices=0)
msm.set(molsys, forcefield='AMBER14', implicit_solvent='OBC1')
print('Physical request:', msm.get(molsys, forcefield=True, implicit_solvent=True))
system = msm.convert(molsys, to_form='openmm.System')
print('Parameterized particles:', system.getNumParticles())

Physical request: ['AMBER14', 'OBC1']


Parameterized particles: 304


:::{note}
:class: dropdown

**Demo Systems Catalog**

This example uses bundled Trp-Cage. See {ref}`Demo Systems <user-foundations-entrance-demo-systems>`.
:::

## Comparing the two solvent variants

Both calculations use the same structure and solute field. The model choice is explicit, and the energies include different solvent responses. These numbers are model potential energies, not binding free energies. Ackredit, when installed, records successful model construction and the actual OpenMM version in the application's session. Applications needing a portable bibliography should capture it separately from OpenMM XML.

In [9]:
for model in ['OBC1', 'OBC2']:
    msm.set(molsys, implicit_solvent=model)
    energy = msm.molecular_mechanics.get_potential_energy(molsys, platform='CPU')
    energy_kj_mol = msm.pyunitwizard.get_value(energy, to_unit='kilojoule/mole')
    print(f'{model}: {float(energy_kj_mol):.3f} kJ/mol')

vacuum_system = msm.convert(molsys, to_form='openmm.System', implicit_solvent=None)
print('Stored choice after the explicit override:', msm.get(molsys, implicit_solvent=True))

OBC1: -2501.906 kJ/mol
OBC2: -2431.238 kJ/mol


Stored choice after the explicit override: OBC2


## Choosing constraints

Constraint policies remain physical names: `None`/`none`, `hbonds` (the default), `all bonds`, and `h angles`. The adapter translates them to the engine's supported controls. OBC calculations accept no cutoff or periodic/nonperiodic cutoff methods and refuse PME/Ewald/LJPME. An absent OBC cutoff setting selects no cutoff even with a box.

Request no constraints explicitly when comparing an unconstrained reference potential:

In [10]:
unconstrained = msm.convert(molsys, to_form='openmm.System', constraints=None)
print('Requested constraints:', unconstrained.getNumConstraints())

Requested constraints: 0


:::{seealso}
:class: dropdown

**Related Tools & References**

- {func}`molsysmt.basic.convert`: Construct a supported external representation using explicit or stored settings.
- {func}`molsysmt.basic.set`: Declare the physical-model request.
- {func}`molsysmt.basic.get`: Inspect the stored request without implying execution.

- {ref}`Get potential energy <Tutorial_Get_potential_energy>`: Calculate potential energy with {func}`molsysmt.molecular_mechanics.get_potential_energy`.
- {ref}`Potential energy minimization <Tutorial_Potential_energy_minimization>`: Minimize potential energy with {func}`molsysmt.molecular_mechanics.potential_energy_minimization`.
:::
